# 🎯 Attention-LSTM Training Notebook

This notebook loads historical price data from all asset classes, prepares time-series sequences, trains an Attention-based LSTM model, and saves it as `pretrained_lstm.pth`.

Make sure:
- You have sufficient `.csv` files with `Close` column under `data/historical-data/<asset-class>/`
- You run this before running the allocation model notebook


## 📦 Block 1: Imports

In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset


## 🧠 Block 2: Attention-Based LSTM Model

In [2]:
class AttentionLSTM(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, output_dim=1):
        super(AttentionLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.attn = nn.Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def attention(self, lstm_output):
        weights = torch.softmax(self.attn(lstm_output).squeeze(-1), dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_output).squeeze(1)
        return context

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        context = self.attention(lstm_out)
        return self.fc(context)


## 📥 Block 3: Load Historical Data from All Asset Classes

In [5]:
def load_all_time_series(data_root='data/historical-data', seq_len=60):
    X, y = [], []
    for root, _, files in os.walk(data_root):
        for file in files:
            if file.endswith('.csv'):
                df = pd.read_csv(os.path.join(root, file))
                if 'Close' not in df.columns or len(df) < seq_len + 1:
                    continue
                series = df['Close'].values[-(seq_len + 1):]
                scaled = MinMaxScaler().fit_transform(series.reshape(-1, 1)).flatten()
                X.append(scaled[:-1])
                y.append(scaled[-1])
    X_tensor = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(-1)
    return X_tensor, y_tensor


## 🔁 Block 4: Train the Model

In [6]:
X, y = load_all_time_series()
dataset = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

model = AttentionLSTM()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

for epoch in range(50):
    total_loss = 0
    for batch_X, batch_y in dataset:
        optimizer.zero_grad()
        output = model(batch_X)
        loss = loss_fn(output, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")


Epoch 0, Loss: 10.7107
Epoch 10, Loss: 1.2011
Epoch 20, Loss: 1.2126
Epoch 30, Loss: 1.0643
Epoch 40, Loss: 1.0055


## 💾 Block 5: Save the Trained Model

In [7]:
torch.save(model.state_dict(), 'pretrained_lstm.pth')
print("Model saved as pretrained_lstm.pth")

Model saved as pretrained_lstm.pth
